## Question 1

You are given a large NumPy array of size 5,000,000 initialized with random values. Compute the following element-wise operation:

\[
f(x) = x^2 + 3x + 5
\]

for each element in the array and convert it into a CUDA kernel using Numba.

Tasks:
1. Compare the performance of CPU and GPU implementations.
2. Modify the CUDA kernel to support both float32 and float64 data types.

In [1]:
# Import libraries
import numpy as np
from numba import cuda
import time

# Size of array
N = 5_000_000

# Create arrays
arr64 = np.random.rand(N)             # float64
arr32 = arr64.astype(np.float32)     # float32

# ---------------- CPU COMPUTATION ----------------
def cpu_compute(arr):
    return arr**2 + 3*arr + 5

start = time.time()
cpu_result = cpu_compute(arr64)
cpu_time = time.time() - start

print("CPU Time:", cpu_time)


# ---------------- CUDA KERNEL (float64) ----------------
@cuda.jit
def gpu_compute_float64(arr, result):
    idx = cuda.grid(1)
    if idx < arr.size:
        x = arr[idx]
        result[idx] = x*x + 3*x + 5


# ---------------- CUDA KERNEL (float32) ----------------
@cuda.jit
def gpu_compute_float32(arr, result):
    idx = cuda.grid(1)
    if idx < arr.size:
        x = arr[idx]
        result[idx] = x*x + 3*x + 5


# GPU configuration
threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block


# ---------------- GPU EXECUTION (float64) ----------------
d_arr64 = cuda.to_device(arr64)
d_result64 = cuda.device_array_like(arr64)

start = time.time()
gpu_compute_float64[blocks_per_grid, threads_per_block](d_arr64, d_result64)
cuda.synchronize()
gpu_result64 = d_result64.copy_to_host()
gpu_time64 = time.time() - start

print("GPU Time (float64):", gpu_time64)


# ---------------- GPU EXECUTION (float32) ----------------
d_arr32 = cuda.to_device(arr32)
d_result32 = cuda.device_array_like(arr32)

start = time.time()
gpu_compute_float32[blocks_per_grid, threads_per_block](d_arr32, d_result32)
cuda.synchronize()
gpu_result32 = d_result32.copy_to_host()
gpu_time32 = time.time() - start

print("GPU Time (float32):", gpu_time32)


# ---------------- VALIDATION ----------------
max_error = np.max(np.abs(cpu_result - gpu_result64))
print("Max Error (float64 CPU vs GPU):", max_error)


# ---------------- PERFORMANCE COMPARISON ----------------
print("\n--- Performance Comparison ---")
print(f"CPU Time         : {cpu_time:.6f} sec")
print(f"GPU Time float64 : {gpu_time64:.6f} sec")
print(f"GPU Time float32 : {gpu_time32:.6f} sec")

CPU Time: 0.04166150093078613
GPU Time (float64): 1.8612937927246094
GPU Time (float32): 0.06921696662902832
Max Error (float64 CPU vs GPU): 1.7763568394002505e-15

--- Performance Comparison ---
CPU Time         : 0.041662 sec
GPU Time float64 : 1.861294 sec
GPU Time float32 : 0.069217 sec


## Conclusion

- CPU execution is faster than GPU for this problem because the operation is simple and GPU overhead (memory transfer and kernel launch) dominates execution time.
- GPU with float32 performs significantly better than float64 due to lower memory usage and faster computation.
- float64 is slower on GPU as it requires higher precision arithmetic.
- The maximum error between CPU and GPU results is extremely small (~1e-15), confirming correctness.

Overall, GPU is beneficial for large-scale and computationally intensive tasks, especially when using float32 precision.



## Question 2

Implement and benchmark a 1-D histogram computation for 1 million random values in Python using Numba.

Tasks:
1. Implement histogram using:
   - Pure Python
   - NumPy
   - Numba (JIT-accelerated)
2. Compare performance of all approaches.
3. Analyze correctness of the results.

Reference:
https://numba.pydata.org/numba-examples/examples/density_estimation/histogram/results.html

In [2]:
import numpy as np
import time
from numba import njit

# ---------------- DATA ----------------
N = 1_000_000
bins = 50

data = np.random.rand(N)


# ---------------- PURE PYTHON ----------------
def histogram_python(data, bins):
    hist = [0] * bins

    for x in data:
        idx = int(x * bins)
        if idx == bins:
            idx -= 1
        hist[idx] += 1

    return hist

start = time.time()
hist_py = histogram_python(data, bins)
time_py = time.time() - start

print("Pure Python Time:", time_py)


# ---------------- NUMPY ----------------
start = time.time()
hist_np, _ = np.histogram(data, bins=bins, range=(0,1))
time_np = time.time() - start

print("NumPy Time:", time_np)


# ---------------- NUMBA ----------------
@njit
def histogram_numba(data, bins):
    hist = np.zeros(bins, dtype=np.int64)

    for i in range(len(data)):
        idx = int(data[i] * bins)
        if idx == bins:
            idx -= 1
        hist[idx] += 1

    return hist

# First call (compilation time included)
hist_nb = histogram_numba(data, bins)

# Second call (actual performance)
start = time.time()
hist_nb = histogram_numba(data, bins)
time_nb = time.time() - start

print("Numba Time:", time_nb)


# ---------------- VALIDATION ----------------
print("\n--- Correctness Check ---")
print("Python vs NumPy:", np.allclose(hist_py, hist_np))
print("Numba vs NumPy :", np.allclose(hist_nb, hist_np))


# ---------------- PERFORMANCE ----------------
print("\n--- Performance Comparison ---")
print(f"Pure Python : {time_py:.6f} sec")
print(f"NumPy       : {time_np:.6f} sec")
print(f"Numba       : {time_nb:.6f} sec")

Pure Python Time: 0.22136640548706055
NumPy Time: 0.015416860580444336
Numba Time: 0.0021309852600097656

--- Correctness Check ---
Python vs NumPy: True
Numba vs NumPy : True

--- Performance Comparison ---
Pure Python : 0.221366 sec
NumPy       : 0.015417 sec
Numba       : 0.002131 sec


## Conclusion



- Pure Python implementation is the slowest due to lack of optimization and interpreted execution.

- NumPy is significantly faster because it uses vectorized operations and optimized C-based implementations.

- Numba achieves performance close to NumPy by compiling Python code into machine code using JIT (Just-In-Time compilation). :

- The first Numba call includes compilation overhead, but subsequent executions are much faster.

- All implementations produce the same result, confirming correctness.

Overall:
- **Pure Python → Slow**
- **NumPy → Fast (baseline)**
- **Numba → Near NumPy speed with custom flexibility**



## Question 3

Write a function `monte_carlo_pi(nsamples)` that estimates the value of π by generating random (x, y) coordinates between 0 and 1 and checking whether they fall inside a unit circle:

\[
x^2 + y^2 < 1
\]

Tasks:
1. Implement the function in pure Python and then using Numba.
2. Compare execution time for 5 million samples.
3. Compute the Speedup Factor (Python Time / Numba Time).
4. Explain why the first execution of the Numba function is slower than subsequent executions.

In [3]:
import random
import time
from numba import njit

# ---------------- PURE PYTHON ----------------
def monte_carlo_pi_python(nsamples):
    inside = 0

    for _ in range(nsamples):
        x = random.random()
        y = random.random()

        if x*x + y*y < 1:
            inside += 1

    return (4 * inside) / nsamples


# ---------------- NUMBA ----------------
@njit
def monte_carlo_pi_numba(nsamples):
    inside = 0

    for i in range(nsamples):
        x = random.random()
        y = random.random()

        if x*x + y*y < 1:
            inside += 1

    return (4 * inside) / nsamples


# ---------------- RUN EXPERIMENT ----------------
N = 5_000_000

# Python timing
start = time.time()
pi_python = monte_carlo_pi_python(N)
time_python = time.time() - start

print("Python PI Estimate:", pi_python)
print("Python Time:", time_python)


# First Numba call (includes compilation)
monte_carlo_pi_numba(N)

# Second call (actual timing)
start = time.time()
pi_numba = monte_carlo_pi_numba(N)
time_numba = time.time() - start

print("\nNumba PI Estimate:", pi_numba)
print("Numba Time:", time_numba)


# ---------------- SPEEDUP ----------------
speedup = time_python / time_numba

print("\n--- Performance ---")
print(f"Python Time : {time_python:.6f} sec")
print(f"Numba Time  : {time_numba:.6f} sec")
print(f"Speedup     : {speedup:.2f}x")

Python PI Estimate: 3.1411536
Python Time: 1.2601120471954346

Numba PI Estimate: 3.1412232
Numba Time: 0.11857795715332031

--- Performance ---
Python Time : 1.260112 sec
Numba Time  : 0.118578 sec
Speedup     : 10.63x


## Conclusion

- The Monte Carlo method estimates π by randomly sampling points and checking how many fall inside a unit circle.

- Pure Python implementation is slow due to interpreted execution and loop overhead.

- Numba significantly improves performance by compiling Python code into optimized machine code.

- The computed π values from both implementations are close to the actual value (~3.1416), confirming correctness.

- Speedup Factor shows how much faster Numba is compared to pure Python.

### Why first Numba execution is slower?

- The first execution includes **JIT compilation overhead**, where Numba translates Python code into machine code.
- Subsequent executions reuse the compiled code, making them much faster.

Overall:
- **Python → Simple but slow**
- **Numba → Fast and efficient for numerical simulations**



## Question 4

You are given a 1D NumPy array representing pixel intensities (values 0–255). You need to increase the brightness of every pixel by 20%, ensuring that no value exceeds 255.

Tasks:
1. Write a function `adjust_brightness(pixel_value)` using the `@vectorize` decorator.
2. Apply this function to an array of 10 million random integers.
3. Modify the decorator to `@vectorize(['int64(int64)'], target='parallel')` and compare execution time.
4. Analyze what happens when a Python list is passed instead of a NumPy array.

In [4]:
import numpy as np
import time
from numba import vectorize

# ---------------- DATA ----------------
N = 10_000_000
pixels = np.random.randint(0, 256, size=N, dtype=np.int64)


# ---------------- NORMAL VECTORIZE ----------------
@vectorize(['int64(int64)'])
def adjust_brightness(x):
    val = int(x * 1.2)
    if val > 255:
        val = 255
    return val


start = time.time()
result_normal = adjust_brightness(pixels)
time_normal = time.time() - start

print("Time (Normal Vectorize):", time_normal)


# ---------------- PARALLEL VERSION ----------------
@vectorize(['int64(int64)'], target='parallel')
def adjust_brightness_parallel(x):
    val = int(x * 1.2)
    if val > 255:
        val = 255
    return val


start = time.time()
result_parallel = adjust_brightness_parallel(pixels)
time_parallel = time.time() - start

print("Time (Parallel Vectorize):", time_parallel)


# ---------------- VALIDATION ----------------
print("\n--- Correctness Check ---")
print("Results same:", np.array_equal(result_normal, result_parallel))


# ---------------- TEST WITH LIST ----------------
pixel_list = list(pixels[:10])  # small sample

try:
    result_list = adjust_brightness(pixel_list)
    print("\nList Input Output:", result_list)
except Exception as e:
    print("\nError with list input:", e)


# ---------------- PERFORMANCE ----------------
print("\n--- Performance Comparison ---")
print(f"Normal Vectorize  : {time_normal:.6f} sec")
print(f"Parallel Vectorize: {time_parallel:.6f} sec")

Time (Normal Vectorize): 0.024944543838500977
Time (Parallel Vectorize): 0.0216677188873291

--- Correctness Check ---
Results same: True

List Input Output: [158 200 255 133 187  55 216 158 100 222]

--- Performance Comparison ---
Normal Vectorize  : 0.024945 sec
Parallel Vectorize: 0.021668 sec


## Conclusion



- The `@vectorize` decorator allows element-wise operations similar to NumPy ufuncs.

- The parallel version (`target='parallel'`) distributes computation across multiple CPU cores, leading to improved performance for large arrays.

- Both implementations produce identical results, confirming correctness.

### List vs NumPy Array

- When a Python list is passed:
  - Numba internally converts it to a NumPy array.
  - This introduces overhead and reduces performance.
  - In some cases, it may also raise type-related errors depending on input.

Overall:
- **Vectorize → Simple element-wise acceleration**
- **Parallel Vectorize → Better performance on large data**
- **NumPy arrays → Preferred input for efficiency**



## Question 5

Write Python code to generate synthetic training data of 100,000 samples, 10 features, and binary labels {-1, +1}. Implement binary logistic regression using gradient descent.

Tasks:
1. Implement using standard NumPy (without Numba).
2. Implement using Numba JIT acceleration.
3. Compare correctness and performance.

In [5]:
import numpy as np
import time
from numba import njit

# ---------------- DATA GENERATION ----------------
np.random.seed(0)

N = 100_000   # samples
D = 10        # features

X = np.random.randn(N, D)
true_w = np.random.randn(D)

# Generate labels {-1, +1}
y = np.sign(X @ true_w + 0.1 * np.random.randn(N))
y[y == 0] = 1


# ---------------- NUMPY LOGISTIC REGRESSION ----------------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def train_numpy(X, y, lr=0.01, epochs=50):
    N, D = X.shape
    w = np.zeros(D)

    for _ in range(epochs):
        z = X @ w
        preds = sigmoid(z)

        # Convert y {-1,1} → {0,1}
        y01 = (y + 1) / 2

        grad = (X.T @ (preds - y01)) / N
        w -= lr * grad

    return w


start = time.time()
w_numpy = train_numpy(X, y)
time_numpy = time.time() - start

print("NumPy Time:", time_numpy)


# ---------------- NUMBA LOGISTIC REGRESSION ----------------
@njit
def sigmoid_nb(z):
    return 1.0 / (1.0 + np.exp(-z))

@njit
def train_numba(X, y, lr=0.01, epochs=50):
    N, D = X.shape
    w = np.zeros(D)

    for _ in range(epochs):
        z = X @ w
        preds = sigmoid_nb(z)

        y01 = (y + 1) / 2

        grad = (X.T @ (preds - y01)) / N
        w -= lr * grad

    return w


# First call (compilation)
train_numba(X, y)

# Actual timing
start = time.time()
w_numba = train_numba(X, y)
time_numba = time.time() - start

print("Numba Time:", time_numba)


# ---------------- CORRECTNESS ----------------
diff = np.linalg.norm(w_numpy - w_numba)

print("\n--- Correctness ---")
print("Weight Difference (L2 norm):", diff)


# ---------------- PERFORMANCE ----------------
speedup = time_numpy / time_numba

print("\n--- Performance ---")
print(f"NumPy Time : {time_numpy:.6f} sec")
print(f"Numba Time : {time_numba:.6f} sec")
print(f"Speedup    : {speedup:.2f}x")

NumPy Time: 0.11755585670471191
Numba Time: 0.15781164169311523

--- Correctness ---
Weight Difference (L2 norm): 0.0

--- Performance ---
NumPy Time : 0.117556 sec
Numba Time : 0.157812 sec
Speedup    : 0.74x


## Conclusion

- Both NumPy and Numba implementations produce identical results, confirming correctness.

- NumPy implementation is faster than Numba in this case.

### Reason:

- NumPy uses highly optimized vectorized operations implemented in low-level C/BLAS libraries.
- The main computations (matrix multiplication and gradient calculation) are already efficient.
- Numba does not significantly accelerate such vectorized operations and may introduce slight overhead.

### Key Insight:

- Numba is most beneficial when optimizing explicit Python loops.
- For already vectorized NumPy code, performance gains are limited or may even be slower.

Overall:
- **NumPy → Best for vectorized operations**
- **Numba → Best for loop-based computations**



## Question 6

Write a CUDA kernel to add two large matrices (A + B = C) of size 1024 × 1024.

Tasks:
1. Implement matrix addition on CPU using NumPy.
2. Implement a CUDA kernel using Numba for GPU computation.
3. Compare correctness and execution time.

In [6]:
import numpy as np
import time
from numba import cuda

# ---------------- DATA ----------------
N = 1024

A = np.random.rand(N, N)
B = np.random.rand(N, N)


# ---------------- CPU IMPLEMENTATION ----------------
start = time.time()
C_cpu = A + B
cpu_time = time.time() - start

print("CPU Time:", cpu_time)


# ---------------- CUDA KERNEL ----------------
@cuda.jit
def matrix_add(A, B, C):
    row, col = cuda.grid(2)

    if row < A.shape[0] and col < A.shape[1]:
        C[row, col] = A[row, col] + B[row, col]


# ---------------- GPU EXECUTION ----------------
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array_like(A)

threads_per_block = (16, 16)
blocks_per_grid_x = (N + threads_per_block[0] - 1) // threads_per_block[0]
blocks_per_grid_y = (N + threads_per_block[1] - 1) // threads_per_block[1]

start = time.time()

matrix_add[(blocks_per_grid_x, blocks_per_grid_y), threads_per_block](d_A, d_B, d_C)
cuda.synchronize()

C_gpu = d_C.copy_to_host()
gpu_time = time.time() - start

print("GPU Time:", gpu_time)


# ---------------- VALIDATION ----------------
max_error = np.max(np.abs(C_cpu - C_gpu))
print("\nMax Error:", max_error)


# ---------------- PERFORMANCE ----------------
print("\n--- Performance Comparison ---")
print(f"CPU Time : {cpu_time:.6f} sec")
print(f"GPU Time : {gpu_time:.6f} sec")

CPU Time: 0.003551006317138672
GPU Time: 0.08968710899353027

Max Error: 0.0

--- Performance Comparison ---
CPU Time : 0.003551 sec
GPU Time : 0.089687 sec


## Conclusion

- Matrix addition of size 1024 × 1024 was implemented using both CPU (NumPy) and GPU (CUDA with Numba).

- The CUDA kernel assigns one thread per matrix element using 2D grid indexing.

- Both implementations produce identical results (very small numerical error), confirming correctness.

### Observations

- GPU execution may not always be faster for simple operations like matrix addition due to:
  - Memory transfer overhead (CPU ↔ GPU)
  - Kernel launch overhead

- GPU becomes beneficial for:
  - Larger matrices
  - More complex computations

Overall:
- **CPU → Efficient for simple operations**
- **GPU → Powerful for large-scale parallel computations**